In [1]:
##use spark_env (Python 3.14.2)
##this was run in local to use pyspark - can run on farm
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as f
import pyspark.sql.types as t
import numpy as np
import sys

In [2]:
# -----------------------
# Start Spark
# -----------------------
spark = SparkSession.builder.appName("locus_clumping").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 15:59:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# -----------------------
# Load your .txt.gz file
# -----------------------
df = spark.read.csv(
    "/path/coloc/GCST90270940_for_eqtl.txt.gz",
    sep="\t",
    header=True,
    inferSchema=True
)

In [4]:
df.head()


Row(variant_id='rs3131972', chr='chr1', pos='817341', other_allele='G', effect_allele='A', effect_allele_frequency=0.161, beta=-0.0085, standard_error=0.0129, p_value=0.5103)

In [5]:
df = df.withColumn("pos", f.col("pos").cast("long"))

In [6]:
# -----------------------
# Rename columns to match expected names
# -----------------------
df = df.withColumnRenamed("chr", "chromosome") \
       .withColumnRenamed("pos", "position") \
       .withColumnRenamed("variant_id", "variantId") \
       .withColumnRenamed("p_value", "pValue")

# Add a dummy studyId (required for partitioning)
df = df.withColumn("studyId", f.lit("study1"))

In [7]:
# -----------------------
# Parameters
# -----------------------
baseline_pvalue_cutoff = 1e-5
distance_cutoff = 250000
pvalue_cutoff = 1e-5
flanking_distance = 100000
neglog_pv_cutoff = -np.log10(pvalue_cutoff)


In [8]:
# -----------------------
# Filter baseline p-values
# -----------------------
df = df.filter(f.col("pValue") <= baseline_pvalue_cutoff)

# Add -log10(p)
df = df.withColumn("negLogPValue", -f.log10(f.col("pValue")))

In [9]:
# -----------------------
# Windows
# -----------------------
w1 = Window.partitionBy("studyId", "chromosome").orderBy("position")

w2 = (
    Window.partitionBy("studyId", "chromosome", "locusStart")
    .orderBy("position")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

w3 = Window.partitionBy(
    "studyId", "chromosome", "locusStart", "locusEnd"
).orderBy(f.col("negLogPValue").desc())


In [10]:
# -----------------------
# Locus breaker logic
# -----------------------
df = df.withColumn("prev_position", f.lag("position").over(w1))

df = df.withColumn("distance", f.col("position") - f.col("prev_position"))

df = df.withColumn(
    "locusStart",
    f.when(
        (f.col("distance") > distance_cutoff) | f.col("distance").isNull(),
        f.col("position"),
    )
)

In [11]:
df = df.withColumn(
    "locusStart",
    f.when(
        f.last(f.col("locusStart") - flanking_distance, True).over(
            w1.rowsBetween(-sys.maxsize, 0)
        ) > 0,
        f.last(f.col("locusStart") - flanking_distance, True).over(
            w1.rowsBetween(-sys.maxsize, 0)
        ),
    ).otherwise(f.lit(0)),
)

df = df.withColumn(
    "locusEnd",
    f.max(f.col("position") + flanking_distance).over(w2)
)

df = df.withColumn("rank", f.rank().over(w3))


In [12]:
# -----------------------
# Keep lead variants only
# -----------------------
df = df.filter(
    (f.col("rank") == 1) & (f.col("negLogPValue") > neglog_pv_cutoff)
)




In [13]:
df

DataFrame[variantId: string, chromosome: string, position: bigint, other_allele: string, effect_allele: string, effect_allele_frequency: double, beta: double, standard_error: double, pValue: double, studyId: string, negLogPValue: double, prev_position: bigint, distance: bigint, locusStart: bigint, locusEnd: bigint, rank: int]

In [14]:
# -----------------------
# Final columns
# -----------------------
df = df.select(
    "variantId",
    "chromosome",
    "position",
    "other_allele",
    "effect_allele",
    "effect_allele_frequency",
    "beta",
    "standard_error",
    "distance",
    "pValue",
    "locusStart",
    "locusEnd"
)

In [ ]:
df.write \
    .option("header", True) \
    .mode("append") \
    .csv("/path/coloc")